# 16. 분류 앙상블 배포 산출물 저장
고정 split에서 초기값이 다른 분류기 5개와 배포용 정규화값·임계값을 저장합니다.

In [ ]:
# [0] 실행 환경과 공통 상수 준비
import os, sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, '..')
import numpy as np
import torch
from src.data import load_for_classification
from src.train import train_classifier
from src.evaluate import predict_proba
from src.model import IDClassifier
from src import config
SPLIT_SEED = 42
MODEL_SEEDS = [0, 1, 2, 3, 4]
MACHINES = ('id_00', 'id_02', 'id_04', 'id_06')
PERCENTILE = 90
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(config.MODEL_DIR, exist_ok=True)
print('실행 장치:', DEVICE)

In [ ]:
# [1] 데이터 분할을 한 번만 고정하고 전역 정규화 기준도 함께 받는다.
X_train, y_train, tests, MIN, MAX = load_for_classification(seed=SPLIT_SEED, return_minmax=True)
print('학습 데이터:', X_train.shape, '라벨:', y_train.shape)
print('전역 MIN/MAX:', MIN, MAX)

In [ ]:
# [2] 데이터는 그대로 두고 모델 초기값만 바꾸어 분류기 5개를 학습·저장한다.
modelLST = []
for seed in MODEL_SEEDS:
    torch.manual_seed(seed)
    model = IDClassifier().to(DEVICE)
    model = train_classifier(model, X_train, y_train, DEVICE)
    model_path = os.path.join(config.MODEL_DIR, f'idclf_{seed}.pth')
    torch.save(model.state_dict(), model_path)
    modelLST.append(model)
    print(f'seed {seed} 모델 저장:', model_path)

In [ ]:
# [3] 앱이 학습 때와 같은 기준으로 정규화하도록 전역 MIN/MAX를 저장한다.
minmaxNP = np.array([MIN, MAX])
np.save(os.path.join(config.MODEL_DIR, 'clf_minmax.npy'), minmaxNP)
print('정규화값 저장:', minmaxNP)

In [ ]:
# [4] 기계별 학습 정상의 앙상블 점수 90퍼센타일을 임계값으로 저장한다.
# 테스트 데이터는 임계값 계산에 쓰지 않으므로 데이터 누수가 없다.
thrLST = []
for i in range(len(MACHINES)):
    maskNP = (y_train == i)
    trainNormalNP = X_train[maskNP]
    scoreSUM = np.zeros(len(trainNormalNP))
    for model in modelLST:
        probaNP = predict_proba(model, trainNormalNP, DEVICE)
        scoreSUM = scoreSUM + (1 - probaNP[:, i])
    ensembleScoreNP = scoreSUM / len(modelLST)
    threshold = np.percentile(ensembleScoreNP, PERCENTILE)
    thrLST.append(threshold)
    print(f'{MACHINES[i]} 임계값: {threshold:.6f}')
thresholdNP = np.array(thrLST)
np.save(os.path.join(config.MODEL_DIR, 'clf_thresholds.npy'), thresholdNP)
print('저장 완료:', thresholdNP)